In [ ]:
from huggingface_hub import login, whoami
whoami()

!pip install wandb -Uq

import os, wandb

# ── dedicated workspace for final runs ────────────────────────────────
WANDB_PROJECT = "mdlm-sft-final"            # new project = fresh workspace
WANDB_GROUP   = "stage2-reasoning"           # optional: groups related runs in the UI

os.environ["WANDB_PROJECT"] = WANDB_PROJECT

wandb.login()


In [ ]:
import tempfile
from pathlib import Path
from huggingface_hub import list_bucket_tree, download_bucket_files


BUCKET = "avgJo3/final-mdlm-sft-artifacts"

PREFIXES: dict[str, str] = {
    "base":      "20260714T050657-0e2e053501df5170/model_",
    "reasoning": "20260715T045644-7d0f95338c3101b3/model_",
}

CHAIN_STEMS = {
    "train":        "train",
    "mix":          "mix",
    "ablation":     "ablation",
    "mix-ablation": "mix-ablation",
}
ROUNDS = ["R0", "R1", "R2", "R3", "R4"]

_MODEL_TEMPDIRS: dict[str, tempfile.TemporaryDirectory] = {}


def _round_dirname(stem: str, k: int) -> str:
    """
    Directory naming convention:
      - k == 0 : always `model_trained_r0` (shared R0, regardless of stem).
      - k >= 1, stem == 'train'    : `model_trained_r{k}`
      - k >= 1, other stems        : `model_{stem}-trained_r{k}`
    """
    if k == 0:
        return "model_trained_r0"
    if stem == "train":
        return f"model_trained_r{k}"
    return f"model_{stem}-trained_r{k}"


def load_model_paths_for(ckpt_alias: str) -> dict[str, dict[str, Path]]:
    if ckpt_alias not in PREFIXES:
        raise KeyError(f"unknown checkpoint alias {ckpt_alias!r}; "
                       f"known aliases: {list(PREFIXES)}")
    prefix = PREFIXES[ckpt_alias]

    files = [
        item for item in list_bucket_tree(BUCKET, prefix=prefix, recursive=True)
        if item.type == "file"
    ]
    if not files:
        raise RuntimeError(f"no files found under prefix: {prefix}")

    tmp = tempfile.TemporaryDirectory(prefix=f"models-{ckpt_alias}-")
    _MODEL_TEMPDIRS[ckpt_alias] = tmp
    tmp_root = Path(tmp.name)

    downloads = [(f, str(tmp_root / f.path)) for f in files]
    download_bucket_files(BUCKET, files=downloads)

    data_root = (tmp_root / prefix).parent

    def resolve_chain(stem: str) -> dict[str, Path]:
        paths = {}
        for k in range(len(ROUNDS)):
            model_dir = data_root / _round_dirname(stem, k)
            assert model_dir.exists(), f"model dir not found: {model_dir}"
            paths[f"R{k}"] = model_dir
        return paths

    return {cid: resolve_chain(stem) for cid, stem in CHAIN_STEMS.items()}


model_paths_by_ckpt: dict[str, dict[str, dict[str, Path]]] = {}
model_paths_by_ckpt["base"]      = load_model_paths_for("base")
model_paths_by_ckpt["reasoning"] = load_model_paths_for("reasoning")

In [ ]:
model_paths_by_ckpt

In [ ]:
from datasets import load_dataset

dd = load_dataset("avgJo3/writingprompts-strat", split='test')
dd = dd.save_to_disk("writingprompts-strat")

In [ ]:
!git clone https://github.com/AvgJoe-cpu/mdlm_sft.git
%cd mdlm_sft
!pip install -e . -q

In [ ]:
import sys
sys.path.insert(0, "/content/mdlm_sft/src")

from mdlm_sft.mdlm.mdlm_gen_v3 import MDLMGenerationConfig, generate_mdlm, run_inference

In [ ]:
import json
from pathlib import Path
from datasets import DatasetDict, load_from_disk


# --- what to run ------------------------------------------------------------
INPUT_DATASET_PATH = "/content/writingprompts-strat/test"
OUTPUT_ROOT        = Path("./generations")        # persistent scratch

GEN_KNOBS = {
    "response_length": 128,
    "num_steps":       128,
    "batch_size":      64,
}


def _write_dataset_dict_marker(chain_dir: Path, round_keys: list[str]) -> None:
    marker = chain_dir / "dataset_dict.json"
    marker.write_text(json.dumps({"splits": round_keys}))


def run_all_chains(
    model_paths_by_ckpt: dict[str, dict[str, dict[str, Path]]],
    input_path: Path,
    output_root: Path,
    knobs: dict,
    skip_existing: bool = True,
) -> dict[str, dict[str, DatasetDict]]:
    out: dict[str, dict[str, DatasetDict]] = {}

    for ckpt_alias, chains in model_paths_by_ckpt.items():
        out[ckpt_alias] = {}

        for chain_id, rounds in chains.items():
            chain_dir = output_root / ckpt_alias / chain_id

            if skip_existing and (chain_dir / "dataset_dict.json").exists():
                print(f"[skip] {ckpt_alias}/{chain_id} — DatasetDict already saved")
                out[ckpt_alias][chain_id] = DatasetDict.load_from_disk(str(chain_dir))
                continue

            print(f"[chain] {ckpt_alias}/{chain_id}")

            for round_key, model_path in rounds.items():
                round_dir = chain_dir / round_key

                if skip_existing and round_dir.exists() and any(round_dir.iterdir()):
                    print(f"  [skip]  {round_key} — output exists")
                    continue

                print(f"  [round] {round_key}   model: {model_path}")
                round_dir.parent.mkdir(parents=True, exist_ok=True)

                cfg = MDLMGenerationConfig(
                    model_name_or_path  = str(model_path),
                    dataset_input_path  = str(input_path),
                    dataset_output_path = str(round_dir),
                    **knobs,
                )
                run_inference(cfg)

            # promote the directory of loose splits into a DatasetDict
            _write_dataset_dict_marker(chain_dir, list(rounds.keys()))
            print(f"  [save]  DatasetDict marker written at {chain_dir}")

            out[ckpt_alias][chain_id] = DatasetDict.load_from_disk(str(chain_dir))

    return out


# --- entry ------------------------------------------------------------------
generated_chains = run_all_chains(
    model_paths_by_ckpt,
    input_path  = INPUT_DATASET_PATH,
    output_root = OUTPUT_ROOT,
    knobs       = GEN_KNOBS,
)

# `generated_chains` mirrors chains_by_ckpt in structure. Feed downstream:
#
#     for chain_id, dd in generated_chains["base"].items():
#         df, U_hats = process_chain(dd)

In [ ]:
for chain_id, dd in generated_chains.items():
  print(dd)

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from huggingface_hub import create_bucket, list_bucket_tree, sync_bucket

BUCKET = "avgJo3/final-mdlm-sft-artifacts"
SAFESPOT_PREFIX = "HE-ood-gens"


def push_generated_to_bucket(
    output_root: str | Path,
    bucket: str = BUCKET,
    prefix: str = SAFESPOT_PREFIX,
    timestamped: bool = True,
) -> str:
    output_root = Path(output_root)
    assert output_root.exists() and output_root.is_dir(), (
        f"output_root does not exist or is not a directory: {output_root}"
    )

    if timestamped:
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
        remote_prefix = f"{prefix}/{stamp}"
    else:
        remote_prefix = prefix

    try:
        create_bucket(bucket)
        print(f"[bucket] created {bucket}")
    except Exception:
        pass

    dest_uri = f"hf://buckets/{bucket}/{remote_prefix}"

    print(f"[sync] {output_root}  →  {dest_uri}")
    plan = sync_bucket(
        source=str(output_root),   # local → bucket direction
        dest=dest_uri,
        verbose=True,
    )
    print(f"[sync] plan summary: {plan.summary()}")

    landed = [
        item for item in list_bucket_tree(bucket, prefix=remote_prefix, recursive=True)
        if item.type == "file"
    ]
    print(f"[sync] {len(landed)} files uploaded under {bucket}/{remote_prefix}")

    return remote_prefix



# safe-spot: push to bucket immediately after generation, before any downstream work
remote_prefix = push_generated_to_bucket(OUTPUT_ROOT)
print(f"[safe-spot] generations available at: {BUCKET}/{remote_prefix}")

# downstream analysis continues on the in-memory dict:
#     for chain_id, dd in generated_chains["base"].items():
#         df, U_hats = process_chain(dd)

In [ ]:
!pip install evaluate bert-score

In [ ]:
from datasets import Dataset, load_dataset

In [ ]:
!pip install mauve-text

In [ ]:
dummy_list = [
    {"idx": 0, "completion": "This is a dummy completion."},
    {"idx": 1, "completion": "Another dummy completion."},
    {"idx": 2, "completion": "Yet another dummy completion."}
]

dummy_compl = [
    {"idx": 0, "completion": "This is a dummy completion."},
    {"idx": 1, "completion": "Another dummy completion."},
    {"idx": 2, "completion": "Yet another dummy completion"}
]

ds = Dataset.from_list(dummy_list)
ds_compl = Dataset.from_list(dummy_compl)
print(ds)

In [ ]:
mauve = evaluate.load("mauve")

# dummy data (reusing ds and ds_compl from before)
pred_texts = ds["completion"]
ref_texts  = ds_compl["completion"]

mauve_result = mauve.compute(
    predictions=pred_texts,
    references=ref_texts,
)

print(mauve_result.mauve)        # scalar — the main score
print(mauve_result)              # full output: mauve, frontier_integral, divergence_curve

In [ ]:
ds = ds.sort("idx")
ds_compl = ds_compl.sort("idx")
assert ds.features["idx"] == ds_compl.features["idx"]

compl_ref  = ds_compl["completion"]
compl_pred = ds["completion"]

results = []
for i in range(len(compl_ref)):
    result = bertscore.compute(
        predictions=[compl_pred[i]],   # ← list, not string
        references=[compl_ref[i]],     # ← list, not string
        lang="en"
    )
    results.append(result)

print(results)

In [ ]:
ds_compl = load_dataset("avgJo3/writingprompts-strat", split="test")
print(ds_compl)

In [ ]:
# compl_ref is fixed across all chains
compl_ref = ds_compl.sort("id")["completion"]

chain_results: dict[str, dict] = {}

for chain_id, dd in generated_chains["base"].items():
    # dd is a DatasetDict with splits R0..R4
    # each split is a Dataset with "idx" and "completion" columns

    round_results: dict[str, list] = {}

    for round_key, round_ds in dd.items():
        round_ds_sorted = round_ds.sort("id")

        # sanity check: same number of examples as reference
        assert len(round_ds_sorted) == len(compl_ref), (
            f"Length mismatch at {chain_id}/{round_key}: "
            f"{len(round_ds_sorted)} vs {len(compl_ref)}"
        )

        compl_pred = round_ds_sorted["completion"]

        result = bertscore.compute(
            predictions=compl_pred,   # full list — one batched call
            references=compl_ref,
            lang="en",
        )
        round_results[round_key] = result   # keys: precision, recall, f1 (lists)
        print(f"  [{chain_id}/{round_key}]  mean F1: {sum(result['f1'])/len(result['f1']):.4f}")

    chain_results[chain_id] = round_results

# chain_results["ablation"]["R0"]["f1"]  → list of per-sample F1 scores

In [ ]:
def evaluate_ckpt_bertscore(
    generated_chains: dict,
    ckpt_alias: str,
    compl_ref: list[str],
    bertscore,
    sort_key: str = "id",
) -> dict[str, dict]:

    chain_results: dict[str, dict] = {}

    for chain_id, dd in generated_chains[ckpt_alias].items():
        round_results: dict[str, dict] = {}

        for round_key, round_ds in dd.items():
            round_ds_sorted = round_ds.sort(sort_key)

            assert len(round_ds_sorted) == len(compl_ref), (
                f"Length mismatch at {chain_id}/{round_key}: "
                f"{len(round_ds_sorted)} vs {len(compl_ref)}"
            )

            compl_pred = round_ds_sorted["completion"]

            result = bertscore.compute(
                predictions=compl_pred,
                references=compl_ref,
                lang="en",
            )
            round_results[round_key] = result
            print(f"  [{ckpt_alias}/{chain_id}/{round_key}]  mean F1: {sum(result['f1'])/len(result['f1']):.4f}")

        chain_results[chain_id] = round_results

    return chain_results

base_results      = evaluate_ckpt_bertscore(generated_chains, "base",      compl_ref, bertscore)
reasoning_results = evaluate_ckpt_bertscore(generated_chains, "reasoning", compl_ref, bertscore)

# access
base_results["mix"]["R0"]["f1"]
reasoning_results["ablation"]["R3"]["precision"]

In [ ]:
import pandas as pd

rows = []
for ckpt_alias, chain_results in [("base", base_results), ("reasoning", reasoning_results)]:
    for chain_id, round_results in chain_results.items():
        for round_key, result in round_results.items():
            for i, (p, r, f) in enumerate(zip(result["precision"], result["recall"], result["f1"])):
                rows.append({
                    "ckpt_alias": ckpt_alias,
                    "chain_id":   chain_id,
                    "round":      round_key,
                    "idx":        i,
                    "precision":  p,
                    "recall":     r,
                    "f1":         f,
                })

df_bertscore = pd.DataFrame(rows)
print(df_bertscore.head())

In [ ]:
df_summary = (
    df_bertscore
    .groupby(["ckpt_alias", "chain_id", "round"], as_index=False)
    [["precision", "recall", "f1"]]
    .mean()
)
print(df_summary)

In [ ]:

import evaluate

# ── 1. Load the BERTScore metric ──────────────────────────────────────────────
bertscore = evaluate.load("bertscore")

# ── 2. Dummy dataset (predictions vs. references) ────────────────────────────
predictions = [
    "The cat sat on the mat.",
    "She enjoys reading books in the evening.",
    "The stock market fell sharply today.",
]

references = [
    "A cat was sitting on a mat.",
    "She likes to read books at night.",
    "Stock prices dropped significantly today.",
]

# ── 3. Compute BERTScore ──────────────────────────────────────────────────────
results = bertscore.compute(predictions=predictions, references=references, lang="en")

# ── 4. Display per-sample results ─────────────────────────────────────────────
print(f"{'#':<5} {'Precision':<12} {'Recall':<12} {'F1':<12}")
print("-" * 45)
for i, (p, r, f) in enumerate(zip(results["precision"], results["recall"], results["f1"])):
    print(f"{i:<5} {p:<12.4f} {r:<12.4f} {f:<12.4f}")

# ── 5. Aggregate (mean) ───────────────────────────────────────────────────────
n = len(predictions)
print("\nMean scores:")
print(f"  Precision : {sum(results['precision']) / n:.4f}")
print(f"  Recall    : {sum(results['recall'])    / n:.4f}")
print(f"  F1        : {sum(results['f1'])        / n:.4f}")

In [ ]:
def evaluate_ckpt_mauve(
    generated_chains: dict,
    ckpt_alias: str,
    compl_ref: list[str],
    mauve,
    sort_key: str = "id",
) -> dict[str, dict]:

    chain_results: dict[str, dict] = {}

    for chain_id, dd in generated_chains[ckpt_alias].items():
        round_results: dict[str, dict] = {}

        for round_key, round_ds in dd.items():
            round_ds_sorted = round_ds.sort(sort_key)

            assert len(round_ds_sorted) == len(compl_ref), (
                f"Length mismatch at {chain_id}/{round_key}: "
                f"{len(round_ds_sorted)} vs {len(compl_ref)}"
            )

            compl_pred = round_ds_sorted["completion"]

            result = mauve.compute(
                predictions=compl_pred,
                references=compl_ref,
            )
            round_results[round_key] = result.mauve   # scalar
            print(f"  [{ckpt_alias}/{chain_id}/{round_key}]  MAUVE: {result.mauve:.4f}")

        chain_results[chain_id] = round_results

    return chain_results


mauve = evaluate.load("mauve")

base_mauve_results      = evaluate_ckpt_mauve(generated_chains, "base",      compl_ref, mauve)
reasoning_mauve_results = evaluate_ckpt_mauve(generated_chains, "reasoning", compl_ref, mauve)

# access
base_mauve_results["mix"]["R0"]           # scalar
reasoning_mauve_results["ablation"]["R3"] # scalar

In [ ]:
rouge = evaluate.load("rouge")

# dummy data (reusing ds and ds_compl from before)
pred_texts = ds["completion"]
ref_texts  = ds_compl["completion"]

rouge_result = rouge.compute(
    predictions=pred_texts,
    references=ref_texts,
    use_aggregator=False,   # ← returns lists, one value per sample
)

print(rouge_result)  # {"rouge1": [...], "rouge2": [...], "rougeL": [...], "rougeLsum": [...]}